In [2]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-groq faiss-cpu pypdf sentence-transformers
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
pdf_path = r"C:\Users\hp\Downloads\DeepLearning_Notes.pdf"   
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"Pages loaded: {len(docs)}")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = splitter.split_documents(docs)
print(f"Chunks created: {len(chunks)}")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
db = FAISS.from_documents(chunks, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 3})
print("FAISS index ready!")
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    api_key="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"   
)
def print_sources(retrieved_docs):
    print("\n\n===== SOURCES USED =====\n")
    for i, doc in enumerate(retrieved_docs):
        page = doc.metadata.get("page", "Unknown")
        preview = doc.page_content[:100].replace("\n", " ")
        print(f"Chunk {i+1}")
        print(f"Page Number: {page}")
        print(f"Preview: {preview}")
        print("-" * 50)
def ask_question(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"""
Answer ONLY if the context contains the exact information.
If not found, respond exactly:
"I could not find this in the document."
Do NOT guess.
Context:
{context}
Question:
{question}
""" 
    response = llm.invoke(prompt)
    print("\n================ ANSWER ================\n")
    print(response.content)
    print_sources(retrieved_docs)
questions = [
    "What is deep learning?",
    "What are neural networks?",
    "Explain backpropagation.",
    "What is overfitting?",
    "What is gradient descent?"
]
for q in questions:
    print("\n\n======================================")
    print("QUESTION:", q)
    ask_question(q)

C:\Users\hp\AppData\Local\Temp\ipykernel_22132\2000851748.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages loaded: 16
Chunks created: 52


C:\Users\hp\AppData\Local\Temp\ipykernel_22132\2000851748.py:40: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index ready!


QUESTION: What is deep learning?

================ ANSWER ================

I could not find this in the document.


===== SOURCES USED =====

Chunk 1
Page Number: 1
Preview: 1 What is a Neural Network? A neural network is a system of connected layers — each layer learns a d
--------------------------------------------------
Chunk 2
Page Number: 2
Preview: 2 Tensors & PyTorch Basics PyTorch is the most popular Deep Learning framework in research and indus
--------------------------------------------------
Chunk 3
Page Number: 6
Preview: 4 Backpropagation & Gradients Backpropagation is how the network learns. After a forward pass produc
--------------------------------------------------


QUESTION: What are neural networks?

================ ANSWER ================

A neural network is a system of connected layers — each layer learns a different level of abstraction from data.


===== SOURCES USED =====

Chunk 1
Page Number: 1
Preview: 1 What is a Neural Network? A 